In [ ]:
# Kaggle Notebook: LSTM & SVM Retraining (38 Features)
# Petunjuk:
# 1. Pastikan Anda sudah mengaktifkan GPU T4 x2 (Penting untuk LSTM!) di pojok kanan atas.
# 2. Upload file 'processed_tropical_features.csv' ke dalam Kaggle.
# 3. Ubah variabel DATA_FILE jika Anda menguploadnya melalui menu "Add Data".
# 4. Copy-Paste seluruh kode ke dalam SATU cell Kaggle dan klik "Run All".

import pandas as pd
import numpy as np
import os
import shutil
import joblib
import json
import time
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
from IPython.display import FileLink, display

# Matikan warning
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Keras / TensorFlow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# ==========================================
# KONFIGURASI
# ==========================================
DATA_FILE = "processed_tropical_features.csv" # Ubah path jika ditaruh di folder /kaggle/input/...
ARTIFACTS_DIR = "agrisense_lstm_svm_models"
AUDIT_FILE = os.path.join(ARTIFACTS_DIR, "tahap4_lstm_svm_audit.json")

os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TARGET_MAP = {
    'target_co2_ppm': 'CO2 (ppm)',
    'target_nee_agrisense': 'Carbon Flux (NEE AgriSense)',
    'target_carbon_potential_score': 'Carbon Potential Score',
    'target_kelembapan_tanah': 'Soil Moisture (%)',
    'target_ph_tanah': 'pH Tanah'
}

# Parameter LSTM
SEQ_LEN = 24       
HORIZON = 24       
BATCH_SIZE = 128
EPOCHS = 15
LEARNING_RATE = 0.001

def create_sequences(data_features, data_targets, seq_len, horizon):
    xs, ys = [], []
    for i in range(len(data_features) - seq_len - horizon + 1):
        x = data_features[i : i + seq_len]
        y = data_targets[i + seq_len + horizon - 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

print("="*60)
print("1. MEMUAT DATASET (38 Fitur)")
print("="*60)
df = pd.read_csv(DATA_FILE)

# Pisahkan target dan fitur
available_targets = [t for t in TARGET_MAP.keys() if t in df.columns]
feature_columns = [c for c in df.columns if c not in TARGET_MAP.keys() and c != 'TIMESTAMP']

feature_df = df[feature_columns].copy()
feature_df.ffill(inplace=True)
feature_df.bfill(inplace=True)

# ==========================================
# TRAINING SVM
# ==========================================
print("\n" + "="*60)
print("2. MULAI TRAINING SVM (Support Vector Machine)")
print("="*60)

# SVM butuh MinMaxScaler
svm_scaler = MinMaxScaler()
X_svm = svm_scaler.fit_transform(feature_df)
joblib.dump(svm_scaler, os.path.join(ARTIFACTS_DIR, "svm_feature_scaler.joblib"))

# Kita pakai 10.000 baris terakhir saja untuk SVM agar tidak memakan waktu berhari-hari
SVM_LIMIT = 10000 
X_svm_train = X_svm[-SVM_LIMIT:]

audit_results = {"SVM": {}, "LSTM": {}}

for target in available_targets:
    target_label = TARGET_MAP[target]
    print(f"-> Training SVM untuk: {target_label}")
    
    y_svm_train = df[target].values[-SVM_LIMIT:]
    
    svm_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
    svm_model.fit(X_svm_train, y_svm_train)
    
    # Save Model
    joblib.dump(svm_model, os.path.join(ARTIFACTS_DIR, f"svm_model_{target_label}.joblib"))
    audit_results["SVM"][target_label] = "Trained with 38 features"

print("SVM Training Selesai!")

# ==========================================
# TRAINING LSTM (Deep Learning)
# ==========================================
print("\n" + "="*60)
print("3. MULAI TRAINING LSTM (Deep Learning Keras)")
print("="*60)

# LSTM butuh StandardScaler
lstm_scaler = StandardScaler()
X_lstm = lstm_scaler.fit_transform(feature_df)
joblib.dump(lstm_scaler, os.path.join(ARTIFACTS_DIR, "lstm_feature_scaler.joblib"))

feature_dim = X_lstm.shape[1]

for target in available_targets:
    target_label = TARGET_MAP[target]
    print(f"\n---> Training LSTM untuk: {target_label} <---")
    
    # Target Scaler
    target_scaler = StandardScaler()
    scaled_target = target_scaler.fit_transform(df[[target]])
    joblib.dump(target_scaler, os.path.join(ARTIFACTS_DIR, f"lstm_target_scaler_{target_label}.joblib"))
    
    # Create Sequences
    print("Membentuk Tensor 3D...")
    X_seq, y_seq = create_sequences(X_lstm, scaled_target.flatten(), SEQ_LEN, HORIZON)
    
    split_idx = int(len(X_seq) * 0.8)
    X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
    y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]
    
    # Build LSTM Model
    model = Sequential([
        LSTM(64, activation='tanh', return_sequences=True, input_shape=(SEQ_LEN, feature_dim)),
        Dropout(0.2),
        LSTM(64, activation='tanh'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(optimizer=optimizer, loss='mse')
    
    # Train
    print("Sedang melatih jaringan saraf...")
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.1, verbose=1)
    
    # Save Model
    model_path = os.path.join(ARTIFACTS_DIR, f"lstm_model_{target_label}_h{HORIZON}.keras")
    model.save(model_path)
    
    # Evaluate
    pred = model.predict(X_test)
    pred_real = target_scaler.inverse_transform(pred)
    actual_real = target_scaler.inverse_transform(y_test.reshape(-1, 1))
    
    r2 = r2_score(actual_real, pred_real)
    print(f"[*] R2 Score: {r2:.4f}")
    audit_results["LSTM"][target_label] = {"R2": float(r2)}

with open(AUDIT_FILE, 'w') as f:
    json.dump(audit_results, f, indent=4)

# ========================================== 
# ZIPPING & DOWNLOAD
# ==========================================
print("\n" + "="*60)
print("TRAINING SELESAI! SEDANG MELAKUKAN ZIPPING...")

ZIP_NAME = "agrisense_lstm_svm_models"
shutil.make_archive(ZIP_NAME, 'zip', ARTIFACTS_DIR)

print("="*60)
print("ZIP SELESAI! Silakan klik link di bawah ini untuk mengunduh:")
display(FileLink(f'{ZIP_NAME}.zip'))
print("="*60)
